# 🧪 Laboratorio 1 — Espectral, centralidad y cascadas en redes que no han visto

**Minicurso LaPSEE · UNESP Ilha Solteira · duración: ~60 min**

En la lección trabajamos con el IEEE 30. Hoy van a aplicar las mismas herramientas a **dos
redes nuevas** — el IEEE 39 (Nueva Inglaterra) y el IEEE 57 — y a una **red sintética**
generada por ustedes. La meta no es repetir la clase: es descubrir que varios resultados
**contradicen la intuición**, y aprender a defenderlos con números.

## Reglas del juego

1. **Trabajen en parejas.** Una persona escribe, la otra predice el resultado *antes* de correr.
2. Los bloques marcados `# ╔══ COMPLETAR` son suyos. Justo debajo hay un **CHEQUEO**
   automático (`assert`): si la celda corre sin error, su código es correcto.
3. Las preguntas 📝 se responden por escrito en la celda correspondiente — **son el
   entregable del laboratorio** junto con la recomendación final.
4. Prohibido seguir adelante con un CHEQUEO en rojo.

| Bloque | Red | Tema | Tiempo |
|---|---|---|---|
| A | IEEE 39 (Nueva Inglaterra) | análisis espectral | 20 min |
| B | IEEE 39 | centralidades en conflicto | 15 min |
| C | IEEE 57 | fallos en cascada | 20 min |
| D (bonus) | sintética ER | ¿la aleatoriedad es mejor diseño? | 10 min |

---
## 0 · Setup (entregado completo — solo correr)

Los tres helpers son los que construimos paso a paso en la Sesión 1: conversión de ramas a
p.u. (con tap), grafo topológico, coordenadas, y la PTDF. Si alguno no les resulta familiar,
es momento de levantar la mano.

In [ ]:
import json
import logging
import warnings

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import pandapower as pp
import pandapower.networks as pn
from scipy.stats import spearmanr

RANDOM_SEED = 2026
rng = np.random.default_rng(RANDOM_SEED)
logging.getLogger("pandapower").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
AZUL, ROJO, GRIS = "#1F3A5F", "#C8102E", "#a0a0a0"


def coords_de_buses(net):
    "{bus: (x, y)} — compatible pandapower 2.x y 3.x."
    if hasattr(net, "bus_geodata") and len(getattr(net, "bus_geodata", [])):
        return {int(b): (float(net.bus_geodata.at[b, "x"]),
                         float(net.bus_geodata.at[b, "y"])) for b in net.bus.index}
    return {int(b): tuple(json.loads(g)["coordinates"][:2])
            for b, g in net.bus["geo"].items()}


def ramas_de_la_red(net):
    "(i, j, x_pu, tipo) por cada línea/trafo, con corrección por tap (S1 §2.1)."
    S = net.sn_mva
    for _, ln in net.line.iterrows():
        vn = net.bus.loc[int(ln.from_bus), "vn_kv"]
        yield int(ln.from_bus), int(ln.to_bus), \
              float(ln.x_ohm_per_km * ln.length_km / (vn**2 / S)), "linea"
    for _, tr in net.trafo.iterrows():
        x = (tr.vk_percent / 100.0) * (S / tr.sn_mva)
        if not np.isnan(tr.tap_pos):
            x *= 1 + (tr.tap_pos - tr.tap_neutral) * tr.tap_step_percent / 100.0
        yield int(tr.hv_bus), int(tr.lv_bus), float(x), "trafo"


def grafo_topologico(net):
    G = nx.Graph(); G.add_nodes_from(sorted(net.bus.index))
    for i, j, _x, _t in ramas_de_la_red(net):
        G.add_edge(i, j)
    return G


def ptdf_de_la_red(net):
    "PTDF (ramas × barras) con slack = ext_grid; ver S1 §5."
    bidx = {b: i for i, b in enumerate(sorted(net.bus.index))}
    n = len(bidx); B = np.zeros((n, n)); edges, bs = [], []
    for i, j, x, _t in ramas_de_la_red(net):
        if abs(x) < 1e-12: continue
        b = 1.0 / x
        B[bidx[i], bidx[j]] -= b; B[bidx[j], bidx[i]] -= b
        B[bidx[i], bidx[i]] += b; B[bidx[j], bidx[j]] += b
        edges.append((bidx[i], bidx[j])); bs.append(b)
    slack = bidx[int(net.ext_grid.bus.iloc[0])]
    resto = [k for k in range(n) if k != slack]
    Binv = np.linalg.inv(B[np.ix_(resto, resto)])
    PTDF = np.zeros((len(edges), n)); pos_de = {k: c for c, k in enumerate(resto)}
    for l, (i, j) in enumerate(edges):
        f = np.zeros(len(resto))
        if i != slack: f[pos_de[i]] += bs[l]
        if j != slack: f[pos_de[j]] -= bs[l]
        PTDF[l, resto] = f @ Binv
    return PTDF


def dibujar_red(net, colores_nodo=None, aristas_rojas=(), titulo="", etiquetas=True):
    "Mapa rápido de la red; colores_nodo: dict {bus: color}; aristas_rojas: set de frozensets."
    pos = coords_de_buses(net); G = grafo_topologico(net)
    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    for u, v in G.edges():
        (x1, y1), (x2, y2) = pos[u], pos[v]
        if frozenset((u, v)) in aristas_rojas:
            ax.plot([x1, x2], [y1, y2], color=ROJO, lw=2.6, zorder=2)
        else:
            ax.plot([x1, x2], [y1, y2], color="lightgray", lw=1.1, zorder=1)
    for b, (x, y) in pos.items():
        c = (colores_nodo or {}).get(b, AZUL)
        ax.scatter(x, y, s=180, c=c, edgecolors="k", zorder=3)
        if etiquetas:
            ax.annotate(str(b), (x, y), ha="center", va="center",
                        fontsize=6.5, color="w", zorder=4)
    ax.set_title(titulo); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    plt.tight_layout(); plt.show()
    return G


print("Setup listo —", "pandapower", pp.__version__)

---
## Bloque A · Análisis espectral del IEEE 39 (Nueva Inglaterra)

El **IEEE 39** modela el sistema de Nueva Inglaterra (EE.UU., años 70): 39 barras, 10
generadores, y una estructura famosa por sus *anillos*. Primero, mírenla.

📝 **Predicción A.0 (antes de correr nada más):** mirando el mapa de la celda siguiente,
¿la red les parece más cohesa o menos cohesa que el IEEE 30 de la clase
($\lambda_2^{IEEE30} = 0.212$)? Apunten su apuesta: $\lambda_2^{IEEE39}$ será **mayor** o
**menor** que 0.212, y por qué.

> *Su respuesta aquí:* ______

In [ ]:
net39 = pn.case39()
G39 = dibujar_red(net39, titulo="IEEE 39 — Nueva Inglaterra (las barras 30-38 son generadores)")
nodos39 = sorted(G39.nodes())
print(f"{G39.number_of_nodes()} barras, {G39.number_of_edges()} aristas, "
      f"grado medio {2*G39.number_of_edges()/G39.number_of_nodes():.2f}")

### A.1 — Construir el Laplaciano y extraer $\lambda_2$

Recuerden de la clase: $L = D - A$, los autovalores salen con `np.linalg.eigh` (ya
ordenados), el número de autovalores ≈ 0 cuenta las componentes conexas, y $\lambda_2$ es la
**conectividad algebraica**.

In [ ]:
# ╔══ COMPLETAR ══ pista: A con nx.to_numpy_array(..., nodelist=nodos39, weight=None);
# ║                L = diag(grados) - A; autovalores y autovectores con np.linalg.eigh(L)
# ✏️  ESCRIBAN SU CÓDIGO AQUÍ (borren la línea del raise al terminar)
raise NotImplementedError("Bloque por completar — vean la pista de arriba")

# ── CHEQUEO automático ────────────────────────────────────────────────────────
assert np.isclose(lambdas[0], 0.0, atol=1e-9), "λ1 debe ser 0 (filas de L suman 0)"
assert int((np.abs(lambdas) < 1e-9).sum()) == 1, "debe haber UN solo autovalor nulo (red conexa)"
assert lambdas.min() > -1e-9, "L es semidefinida positiva: nada negativo"
assert 0.05 < lam2 < 0.10, "λ2 fuera del rango esperado: revisen la construcción de L"
print(f"✅ CHEQUEO OK — λ2(IEEE 39) = {lam2:.4f}   vs   λ2(IEEE 30) = 0.2121")

💡 **Resultado contraintuitivo nº1.** El IEEE 39 tiene *más barras y más aristas* que el
IEEE 30... y sin embargo $\lambda_2$ es casi **3 veces menor**: es mucho más fácil de partir.
El tamaño no compra cohesión — la *forma* sí. Nueva Inglaterra está hecha de anillos largos
colgados de pocos corredores; el IEEE 30 es más "malla".

📝 **Pregunta A.1:** ¿acertó su predicción? Si no, ¿qué rasgo del mapa los engañó?

> *Su respuesta aquí:* ______

### A.2 — ¿Por dónde se parte? La bipartición de Fiedler

Coloreen cada barra según el **signo** de su componente en el vector de Fiedler y cuenten
las aristas "frontera" (las que conectan signos opuestos).

In [ ]:
# ╔══ COMPLETAR ══ pista: signo = {bus: fiedler[k] >= 0}; una arista (u,v) es frontera
# ║                si signo[u] != signo[v]
# ✏️  ESCRIBAN SU CÓDIGO AQUÍ (borren la línea del raise al terminar)
raise NotImplementedError("Bloque por completar — vean la pista de arriba")

# ── CHEQUEO automático ────────────────────────────────────────────────────────
assert len(frontera) == 3, f"deberían ser exactamente 3 aristas frontera, tienen {len(frontera)}"
print(f"✅ CHEQUEO OK — la frontera espectral son solo {len(frontera)} aristas: {frontera}")
print(f"   Tamaños de las comunidades: {sum(signo.values())} vs {len(nodos39)-sum(signo.values())}")

dibujar_red(net39,
            colores_nodo={b: (ROJO if s else AZUL) for b, s in signo.items()},
            aristas_rojas=set(map(frozenset, frontera)),
            titulo="Bipartición de Fiedler del IEEE 39 — 3 aristas sostienen la unión")

💡 Toda Nueva Inglaterra cuelga de **3 líneas**. Si esas tres se congestionan a la vez, el
sistema queda partido en dos zonas de precio (Sesión 2) — y si se *disparan* a la vez, en
dos islas eléctricas.

### A.3 — El experimento del planificador: ¿dónde vale más una línea?

Tres cirugías sobre el grafo (código entregado — su trabajo es **predecir antes de correr**
y luego explicar):

1. **quitar** la arista frontera (13, 14);
2. **agregar** una línea entre los dos extremos del vector de Fiedler (las barras más
   "alejadas" espectralmente);
3. **agregar** una línea entre dos barras de la *misma* comunidad.

📝 **Predicción A.3:** ordenen de mayor a menor el $\lambda_2$ resultante de (1), (2), (3).

> *Su respuesta aquí:* ______

In [ ]:
def lam2_de(G):
    nd = sorted(G.nodes())
    A_ = nx.to_numpy_array(G, nodelist=nd, weight=None)
    return float(np.linalg.eigvalsh(np.diag(A_.sum(1)) - A_)[1])

# (1) quitar la arista frontera (13, 14)
G_sin = G39.copy(); G_sin.remove_edge(13, 14)

# (2) agregar línea entre los extremos del Fiedler
b_min, b_max = nodos39[int(np.argmin(fiedler))], nodos39[int(np.argmax(fiedler))]
G_puente = G39.copy(); G_puente.add_edge(b_min, b_max)

# (3) agregar línea dentro de la misma comunidad (entre dos no-vecinas)
comunidad = [b for b in nodos39 if signo[b]]
a_, b_ = next((a, b) for a in comunidad for b in comunidad
              if a < b and not G39.has_edge(a, b))
G_local = G39.copy(); G_local.add_edge(a_, b_)

tabla = pd.DataFrame({
    "escenario": ["red original", f"(1) quitar frontera (13,14)",
                  f"(2) puente {b_min}–{b_max} (extremos Fiedler)",
                  f"(3) línea local {a_}–{b_}"],
    "lambda_2": [lam2, lam2_de(G_sin), lam2_de(G_puente), lam2_de(G_local)],
})
tabla["Δ vs original"] = (tabla["lambda_2"] - lam2).round(4)
display(tabla.round(4))
print(f"¿Sigue conexa sin (13,14)? {nx.is_connected(G_sin)}")

💡 **Resultado contraintuitivo nº2.** Comparen los dos *agregados*: la línea puente entre
extremos espectrales sube $\lambda_2$ ~**17 veces más** que la línea local — misma
inversión, retorno estructural radicalmente distinto. Y quitar UNA arista frontera reduce la
cohesión a la mitad sin desconectar nada: la red queda entera pero frágil. **El vector de
Fiedler es, literalmente, un mapa de dónde construir.**

📝 **Pregunta A.3:** si fueran el regulador y solo pudieran financiar una línea nueva en
Nueva Inglaterra, ¿entre qué dos barras? Justifiquen con $\Delta\lambda_2$ *y* con el mapa.

> *Su respuesta aquí:* ______

---
## Bloque B · Centralidades en conflicto: ¿a quién proteger?

Tres definiciones de "barra importante", tres respuestas. Sobre el mismo IEEE 39:

| Métrica | Pregunta que responde |
|---|---|
| grado $k_i$ | ¿cuántas líneas llegan aquí? |
| betweenness topológica | ¿cuántos caminos mínimos me atraviesan? |
| betweenness **eléctrica** (PTDF) | ¿cuánto perturban MIS inyecciones los flujos del sistema? |

### B.1 — Calcular las tres

La PTDF ya está implementada en el setup (`ptdf_de_la_red`). La betweenness eléctrica de la
barra $k$ es la suma $\sum_\ell |\mathrm{PTDF}_{\ell k}|$ **después** de anular las entradas
menores a `1e-3` (anti-ruido), como en la clase.

In [ ]:
PTDF39 = ptdf_de_la_red(net39)
print("PTDF:", PTDF39.shape, "(ramas × barras)")

# ╔══ COMPLETAR ══ pista: grado con dict(G39.degree()); topológica con
# ║   nx.betweenness_centrality(G39, normalized=False); eléctrica: copien |PTDF|,
# ║   pongan en 0 lo < 1e-3 y sumen por columnas (axis=0)
# ✏️  ESCRIBAN SU CÓDIGO AQUÍ (borren la línea del raise al terminar)
raise NotImplementedError("Bloque por completar — vean la pista de arriba")

# ── CHEQUEO automático ────────────────────────────────────────────────────────
assert len(btw_elec) == 39 and (btw_elec >= 0).all(), "la eléctrica debe ser ≥ 0 en las 39 barras"
assert btw_topo.idxmax() != btw_elec.idxmax(), \
    "¡los dos rankings deberían coronar a barras DISTINTAS! Revisen el cálculo."
print(f"✅ CHEQUEO OK — top topológica: bus {btw_topo.idxmax()} | "
      f"top eléctrica: bus {btw_elec.idxmax()}")

### B.2 — El careo (código entregado)

In [ ]:
rho_s = spearmanr(btw_topo, btw_elec).statistic

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(btw_topo, btw_elec, s=55, c=AZUL, edgecolors="k")
for b in nodos39:
    axes[0].annotate(str(b), (btw_topo[b], btw_elec[b]),
                     textcoords="offset points", xytext=(4, 3), fontsize=7)
axes[0].set_xlabel("betweenness topológica"); axes[0].set_ylabel("betweenness eléctrica")
axes[0].set_title(f"Spearman = {rho_s:+.2f}  (¡miren el signo!)")

top = pd.DataFrame({"topológica": btw_topo.nlargest(5).index,
                    "eléctrica": btw_elec.nlargest(5).index})
axes[1].axis("off")
axes[1].table(cellText=top.values, colLabels=["top-5 topológica", "top-5 eléctrica"],
              loc="center", cellLoc="center").scale(1, 1.8)
axes[1].set_title("Los dos podios no comparten ni una barra")
plt.tight_layout(); plt.show()

print("Grado de las barras del top-5 eléctrico:",
      grado[btw_elec.nlargest(5).index].tolist())

💡 **Resultado contraintuitivo nº3.** La correlación no solo no es 1: **es negativa**. Y el
top-5 eléctrico son las barras 33–37... que tienen **grado 1** — ¡son hojas! Son las barras
de los generadores: cada MW que inyectan debe atravesar media red para llegar a la
referencia, perturbando muchas líneas a su paso. La betweenness topológica pregunta *"¿soy
un cruce de caminos?"*; la eléctrica pregunta *"¿mis inyecciones estresan al sistema?"* —
preguntas distintas, rankings opuestos.

📝 **Pregunta B:** tienen presupuesto para blindar la seguridad física de UNA subestación y,
por separado, para reforzar la evacuación de UN generador. ¿Qué métrica usan para cada
decisión y qué barra eligen en cada caso?

> *Su respuesta aquí:* ______

---
## Bloque C · Cascadas en el IEEE 57: ¿quién es realmente peligroso?

Cambiamos a la tercera red del día. El **IEEE 57** (sistema de la zona de Carolina, EE.UU.)
tiene 57 barras y una estructura más enmarañada. La función de cascada de Motter–Lai es la
de la clase, con una mejora: el daño se mide por la **componente gigante** superviviente
(estar "vivo" pero aislado del sistema no sirve de nada).

In [ ]:
net57 = pn.case57()
G57 = dibujar_red(net57, titulo="IEEE 57", etiquetas=True)
nodos57 = sorted(G57.nodes())
grado57 = pd.Series(dict(G57.degree())).reindex(nodos57)
btw57 = pd.Series(nx.betweenness_centrality(G57, normalized=False)).reindex(nodos57)
print(f"top grado: bus {grado57.idxmax()} (k={grado57.max()}) | "
      f"top betweenness: bus {btw57.idxmax()}  <- ¡no son la misma barra!")


def cascada_motter_lai(G_full, atacados, alpha=1.2, max_iter=100):
    "Cascada de la clase + daño medido sobre la componente gigante final."
    todos = list(G_full.nodes())
    L0 = pd.Series(nx.betweenness_centrality(G_full, normalized=False))
    C = alpha * L0                                     # capacidad fija
    vivos = set(todos) - set(atacados)
    en_cascada = 0
    for _ in range(max_iter):
        H = G_full.subgraph(vivos)
        if H.number_of_edges() == 0:
            break
        L = nx.betweenness_centrality(H, normalized=False)
        sobre = [v for v in vivos if L.get(v, 0.0) > C[v] + 1e-12]
        if not sobre:
            break
        vivos -= set(sobre); en_cascada += len(sobre)
    H = G_full.subgraph(vivos)
    gigante = max((len(c) for c in nx.connected_components(H)), default=0)
    return {"frac_gigante": gigante / len(todos), "en_cascada": en_cascada}

### C.1 — Análisis N−1: el mapa de riesgo

Recorran las 57 barras: ataquen cada una (sola, con $\alpha = 1.2$) y guarden la fracción de
red que se pierde: $1 - $ `frac_gigante`.

In [ ]:
# ╔══ COMPLETAR ══ pista: un dict/Series {bus: 1 - cascada_motter_lai(G57,[bus],1.2)["frac_gigante"]}
# ║                recorriendo nodos57 (tarda ~20 s: 57 cascadas)
# ✏️  ESCRIBAN SU CÓDIGO AQUÍ (borren la línea del raise al terminar)
raise NotImplementedError("Bloque por completar — vean la pista de arriba")

# ── CHEQUEO automático ────────────────────────────────────────────────────────
assert len(extension) == 57 and extension.between(0, 1).all()
b_peor = int(extension.idxmax())
assert b_peor != int(grado57.idxmax()), \
    "¡la barra más peligrosa NO debería ser la de mayor grado! Revisen el bucle."
print(f"✅ CHEQUEO OK — la barra más peligrosa es la {b_peor} "
      f"(pierde {extension.max():.0%} de la red), y NO la de mayor grado "
      f"(bus {grado57.idxmax()}, que pierde {extension[grado57.idxmax()]:.0%})")

ext_ord = extension.sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.bar(ext_ord.index.astype(str), ext_ord.values,
       color=[ROJO if v > 0.4 else AZUL for v in ext_ord.values])
ax.set_xlabel("bus atacado"); ax.set_ylabel("fracción de red perdida")
ax.set_title("N−1 con cascada (α = 1.2) — rojo: el fallo arrastra a >40% del sistema")
ax.tick_params(axis="x", labelsize=6.5)
plt.tight_layout(); plt.show()

### C.2 — Tres estrategias de ataque, una curva de robustez

Barran $\alpha \in [1.0, 1.6]$ para tres ataques: a la barra de **máxima betweenness**, a la
de **máximo grado**, y a la barra **53** (elegida al azar antes de clase). Grafiquen la
fracción de componente gigante superviviente vs. $\alpha$.

📝 **Predicción C.2:** ordenen las tres estrategias de más a menos dañina (para $\alpha$
intermedio). ¿Dónde queda el azar?

> *Su respuesta aquí:* ______

In [ ]:
alphas = np.linspace(1.0, 1.6, 13)
objetivos = {"máx. betweenness": int(btw57.idxmax()),
             "máx. grado": int(grado57.idxmax()),
             "aleatorio (bus 53)": 53}

# ╔══ COMPLETAR ══ pista: para cada estrategia, una lista de
# ║   cascada_motter_lai(G57, [bus], a)["frac_gigante"] recorriendo alphas
# ✏️  ESCRIBAN SU CÓDIGO AQUÍ (borren la línea del raise al terminar)
raise NotImplementedError("Bloque por completar — vean la pista de arriba")

# ── CHEQUEO automático ────────────────────────────────────────────────────────
assert set(curvas) == set(objetivos) and all(len(v) == len(alphas) for v in curvas.values())
medias = {t: float(np.mean(v)) for t, v in curvas.items()}
assert medias["máx. betweenness"] < medias["aleatorio (bus 53)"], "btw debería ser el peor ataque"
print("✅ CHEQUEO OK — daño medio por estrategia:",
      {t: round(1 - m, 2) for t, m in medias.items()})

fig, ax = plt.subplots(figsize=(9.5, 5.2))
for (tag, vals), c in zip(curvas.items(), [ROJO, AZUL, GRIS]):
    ax.plot(alphas, vals, "o-", color=c, label=f"{tag} (bus {objetivos[tag]})")
ax.set_xlabel("tolerancia α"); ax.set_ylabel("fracción de componente gigante")
ax.set_title("Robustez del IEEE 57 ante tres estrategias de ataque")
ax.legend(); plt.tight_layout(); plt.show()

💡 **Resultado contraintuitivo nº4.** Para $\alpha$ intermedio, atacar a la barra de **mayor
grado** hace *menos* daño que el fallo del bus 53 elegido al azar. El grado mide cuántas
líneas llegan; la cascada se alimenta de cuánto **flujo de caminos** se redistribuye — y eso
lo captura la betweenness, no el grado. Noten además los **saltos** de las curvas: la
robustez es un fenómeno de umbral (lo vimos en clase), no compren tolerancia "de a poquito".

📝 **Pregunta C:** son la autoridad de planificación del IEEE 57 y pueden blindar UNA barra
(capacidad infinita). ¿Cuál blindan y por qué? ¿Cambia su respuesta si el adversario es la
naturaleza (fallos aleatorios) en lugar de un atacante informado?

> *Su respuesta aquí:* ______

---
## Bloque D (bonus) · ¿Y si la red fuera aleatoria?

Pregunta de diseño: el IEEE 57 tiene 57 barras y 78 ramas. ¿Qué pasaría si conectáramos esas
**mismas 78 líneas al azar** (modelo Erdős–Rényi)? ¿Sería mejor o peor diseño?

In [ ]:
G_er = nx.gnm_random_graph(57, 78, seed=RANDOM_SEED)
print(f"ER(57, 78): ¿nace conexa? {nx.is_connected(G_er)}")
print(f"  componentes: {[len(c) for c in sorted(nx.connected_components(G_er), key=len, reverse=True)]}")
G_er_gig = G_er.subgraph(max(nx.connected_components(G_er), key=len)).copy()
print(f"  trabajamos con la componente gigante: {G_er_gig.number_of_nodes()} nodos")

💡 **Resultado contraintuitivo nº5 (gratis, antes de calcular nada).** Con el *mismo
presupuesto de líneas*, la red aleatoria **ni siquiera nace conexa** — deja barras huérfanas.
Las redes reales no son aleatorias: cada línea fue puesta para garantizar, primero que nada,
que todos queden conectados.

### D.1 — El careo final: real vs. aleatoria

In [ ]:
# ╔══ COMPLETAR ══ pista: para cada grafo: lam2_de(G) y el daño del ataque dirigido
# ║   1 - cascada_motter_lai(G, [bus de máx betweenness], 1.2)["frac_gigante"]
# ✏️  ESCRIBAN SU CÓDIGO AQUÍ (borren la línea del raise al terminar)
raise NotImplementedError("Bloque por completar — vean la pista de arriba")

# ── CHEQUEO automático ────────────────────────────────────────────────────────
assert resultados.shape[0] == 2 and resultados["lambda_2"].between(0, 2).all()
display(resultados.round(3))
dano_real = float(resultados.iloc[0, 2]); dano_er = float(resultados.iloc[1, 2])
assert dano_real > dano_er, "sorpresa: la red REAL debería sufrir más el ataque dirigido"
print(f"✅ CHEQUEO OK — daño en la real: {dano_real:.0%} vs. en la aleatoria: {dano_er:.0%}")

💡 **Resultado contraintuitivo nº6.** Las dos redes tienen $\lambda_2$ parecidos... pero ante
el ataque dirigido **la red real pierde mucho más** que la aleatoria. La razón: el diseño
real (radial-encadenado, dictado por la geografía y el costo) **concentra** la betweenness en
unos pocos corredores; el cableado aleatorio la reparte. El precio de la eficiencia económica
es un talón de Aquiles estructural — exactamente la tesis de Motter & Lai.

📝 **Pregunta D:** ¿significa esto que deberíamos cablear las redes "más al azar"? ¿Qué deja
afuera esta comparación? (Pista: ¿cuánto mide un km de línea aleatoria entre dos barras
cualesquiera de Carolina?)

> *Su respuesta aquí:* ______

---
## 🏁 Cierre y entregable

Los seis resultados que acaban de demostrar con sus propias manos:

| # | Resultado | Dónde lo vieron |
|---|---|---|
| 1 | Más grande y con más líneas ≠ más coheso ($\lambda_2^{39} \ll \lambda_2^{30}$) | A.1 |
| 2 | Una línea bien ubicada vale ~17× más que una mal ubicada (en $\Delta\lambda_2$) | A.3 |
| 3 | Los rankings topológico y eléctrico se correlacionan **negativo**; las hojas-generador son las más centrales eléctricamente | B.2 |
| 4 | Atacar al de mayor grado daña menos que un fallo aleatorio | C.2 |
| 5 | Con las mismas líneas, la red aleatoria ni siquiera nace conexa | D |
| 6 | La red real es más frágil al ataque dirigido que la aleatoria | D.1 |

**Entregable (una página):** las respuestas 📝 de A–D más un párrafo final respondiendo:
*"¿Qué métrica única reportarían a un gerente que pregunta cuál es la barra más importante
de la red — y qué le advertirían sobre esa cifra?"*